In [1]:
# ============================================================
# YOLO FINAL PIPELINE (VOC) — COMPLETE + CORRECT + COMPARISON READY
# ============================================================

# ---------------- INSTALL ----------------
!pip install -q ultralytics

# ---------------- IMPORTS ----------------
import time, glob, shutil
import pandas as pd
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import xml.etree.ElementTree as ET
import torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
# ---------------- SETTINGS ----------------
OUTPUT_DIR = "outputs/yolo"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

VOC_ROOT = "/kaggle/input/datasets/huanghanchina/pascal-voc-2012/VOC2012"
YOLO_ROOT = "/kaggle/working/voc_yolo"

EPOCHS = 5

print("="*60)
print("YOLO PIPELINE STARTED")
print("="*60)


YOLO PIPELINE STARTED


In [3]:
# ============================================================
# STEP 1: CONVERT VOC → YOLO
# ============================================================

classes = [
    "aeroplane","bicycle","bird","boat","bottle","bus","car","cat",
    "chair","cow","diningtable","dog","horse","motorbike","person",
    "pottedplant","sheep","sofa","train","tvmonitor"
]

class2idx = {c:i for i,c in enumerate(classes)}

def convert_box(size, box):
    w,h = size
    x1,y1,x2,y2 = box
    xc = (x1+x2)/2/w
    yc = (y1+y2)/2/h
    bw = (x2-x1)/w
    bh = (y2-y1)/h
    return xc,yc,bw,bh

def convert_dataset(split):
    ids = open(f"{VOC_ROOT}/ImageSets/Main/{split}.txt").read().strip().split()

    for img_id in ids:
        xml_file = f"{VOC_ROOT}/Annotations/{img_id}.xml"
        img_file = f"{VOC_ROOT}/JPEGImages/{img_id}.jpg"

        label_file = f"{YOLO_ROOT}/labels/{split}/{img_id}.txt"
        img_out = f"{YOLO_ROOT}/images/{split}/{img_id}.jpg"

        Path(label_file).parent.mkdir(parents=True, exist_ok=True)
        Path(img_out).parent.mkdir(parents=True, exist_ok=True)

        tree = ET.parse(xml_file)
        root = tree.getroot()

        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)

        with open(label_file, "w") as f:
            for obj in root.findall("object"):
                cls = obj.find("name").text
                if cls not in class2idx:
                    continue

                cls_id = class2idx[cls]

                b = obj.find("bndbox")
                box = [
                    float(b.find("xmin").text),
                    float(b.find("ymin").text),
                    float(b.find("xmax").text),
                    float(b.find("ymax").text)
                ]

                xc,yc,bw,bh = convert_box((w,h), box)
                f.write(f"{cls_id} {xc} {yc} {bw} {bh}\n")

        shutil.copy(img_file, img_out)

print("Converting VOC to YOLO format...")
convert_dataset("train")
convert_dataset("val")


Converting VOC to YOLO format...


In [4]:


 
 
# ============================================================
# STEP 2: CREATE YAML
# ============================================================

yaml_text = f"""
path: {YOLO_ROOT}

train: images/train
val: images/val

names:
"""

for i, c in enumerate(classes):
    yaml_text += f"  {i}: {c}\n"

with open("voc.yaml", "w") as f:
    f.write(yaml_text)

print("YAML created")

# ============================================================
# STEP 3: TRAIN YOLO
# ============================================================

model = YOLO("yolov8m.pt")

print("Training YOLO...")

start_train = time.time()

model.train(
    data="voc.yaml",
    epochs=EPOCHS,
    imgsz=400,
    batch=8,
    device=0
)

train_time = time.time() - start_train

# ============================================================
# STEP 4: EVALUATION
# ============================================================

print("Evaluating YOLO...")
metrics = model.val()

map50 = metrics.box.map50
map5095 = metrics.box.map

# ============================================================
# STEP 5: SPEED (CORRECT VERSION)
# ============================================================

print("Measuring YOLO inference speed...")

def measure_yolo_speed(model, image_paths, num_images=50):
    model.eval()
    times = []

    for i, img_path in enumerate(image_paths):
        if i >= num_images:
            break

        # warmup
        if i < 5:
            _ = model(img_path)
            continue

        torch.cuda.synchronize()
        start = time.time()

        _ = model(img_path)

        torch.cuda.synchronize()
        times.append(time.time() - start)

    avg = np.mean(times)
    return avg*1000, 1/avg

image_list = glob.glob(f"{YOLO_ROOT}/images/val/*.jpg")

latency_ms, fps = measure_yolo_speed(model, image_list)

print("Latency (ms):", round(latency_ms,2))
print("FPS:", round(fps,2))

# ============================================================
# STEP 6: SAVE RESULTS
# ============================================================

pd.DataFrame([{
    "epoch": EPOCHS,
    "map50": map50,
    "map5095": map5095,
    "epoch_time_sec": train_time / EPOCHS
}]).to_csv(f"{OUTPUT_DIR}/metrics.csv", index=False)

pd.DataFrame([{
    "latency_ms": latency_ms,
    "fps": fps
}]).to_csv(f"{OUTPUT_DIR}/speed.csv", index=False)

# ============================================================
# STEP 7: VISUALIZATION
# ============================================================

print("Saving predictions...")

model.predict(
    source=f"{YOLO_ROOT}/images/val",
    save=True,
    project=OUTPUT_DIR,
    name="predictions"
)

print("="*60)
print("YOLO PIPELINE COMPLETE")
print("Results saved in:", OUTPUT_DIR)
print("="*60)

YAML created
Training YOLO...
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=voc.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=400, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspec